In [1]:
import os
import re
import json
import math
from pathlib import Path
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# PDF ingestion
from pypdf import PdfReader

# Embeddings
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer
from gensim.models import Word2Vec

# (Optional) local generator
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


# Distributed computing system. 
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession 
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, 
    LongType, FloatType, ArrayType
)

In [2]:
@dataclass
class RAGConfig:
    pdf_path: str = "course_material"

    # Chunking (simple word-based chunking)
    chunk_size_words: int = 200
    chunk_overlap_words: int = 50

    # Retrieval
    top_k: int = 4
    similarity_metric: str = "cosine"   # {"cosine", "euclidean"}
    embedding_strategy: str = "sbert"   # {"tfidf", "sbert"}

    # Embedding models
    sbert_model_name: str = "sentence-transformers/all-MiniLM-L6-v2"

    # Generator (fallback: small, CPU-friendly)
    gen_model_name: str = "google/flan-t5-small"
    max_new_tokens: int = 256

cfg = RAGConfig()

In [3]:
# Init Park
if SparkContext._active_spark_context:
    SparkContext._active_spark_context.stop()
    print("SparkContext existant arrêté.")
    
conf = SparkConf() \
    .setAppName("IngestData") \
    .setMaster("local[*]")

sc = SparkContext(conf=conf)
print("Spark version:", sc.version)

Spark version: 3.3.2


# Quelques fonctions
- sc.parallelize(c, numSlices=None)
    Elle prend deux paramètres :
    - 1. c — la collection (obligatoire)
    N'importe quelle collection Python itérable : list, range, tuple, string, dict, etc

    - 2. numSlices — le nombre de partitions (optionnel)
    Contrôle en combien de morceaux les données sont découpées. Si non spécifié, Spark décide automatiquement selon le nombre de cœurs disponibles.

    - Ce qu'elle retourne
    Un RDD — sur lequel vous pouvez ensuite chaîner des transformations (map, filter, flatMap...) ou des actions (collect, reduce, count...).

The pipeline must keep traceability metadata for every extracted unit, including at least the source PDF
name, page number, chunk identifier, and word offsets when available. This metadata is required later for
citation support and error analysis.

# Module 1 — Ingest: extract text from a PDF → page-level text (+ metadata for citations)

## Converted PDF to page-level raw text

In [4]:
def get_paths_pdf_files(folder_path: str) -> List[str]:
    """
    Browse a folder and get path from each PDF files.

    Parameters
    ----------
    folder_path : str
        Path to the folder containing the PDF files.

    Returns
    -------
    results : list of string
    """

    folder = Path(folder_path)

    # return paths all pdf files in folder path
    return list( folder.glob("*.pdf") )

# --- quick sanity check (should not error once implemented) ---
pdf_paths = get_paths_pdf_files(cfg.pdf_path)
rdd = sc.parallelize(pdf_paths)

In [5]:
print(pdf_paths)

[PosixPath('course_material/1-intro.pdf'), PosixPath('course_material/2-spark.pdf'), PosixPath('course_material/3-stream_proc.pdf'), PosixPath('course_material/4-stream_algos.pdf'), PosixPath('course_material/5-nosql.pdf'), PosixPath('course_material/6-parallel.pdf'), PosixPath('course_material/9780521192248_excerpt.pdf'), PosixPath('course_material/A Neural Network Playground.pdf'), PosixPath('course_material/a_Big Data, Data Mining, and Machine Learning ( PDFDrive.com )-min.pdf'), PosixPath('course_material/a_ScalableMachineLearningAlgorithmsforBigDataAnalytics-ChallengesandOpportunities.pdf'), PosixPath('course_material/a_the-big-book-of-machine-learning-use-cases.pdf'), PosixPath('course_material/a_TowardScalableSystemsForBigDataAnalytics.pdf'), PosixPath('course_material/Finding Similar Items.pdf'), PosixPath('course_material/Gama_ACMCS_AdaptationCD_accepted (1).pdf'), PosixPath('course_material/Gama_ACMCS_AdaptationCD_accepted.pdf'), PosixPath('course_material/intro.pdf'), PosixP

### Différence entre map et flatMap
La différence fondamentale
map → 1 élément entrant = 1 élément sortant
flatMap → 1 élément entrant = N éléments sortants (la liste résultante est "aplatie")

In [6]:
def extract_pages_from_pdf(pdf_path: str) -> List[Dict]:
    # Read pdf file
    reader = PdfReader(pdf_path)

    # For each page
    pages = list()
    for i in range( len(reader.pages) ):
        # Extract the text
        page = {
            "pdf_name" : Path(pdf_path).name,
            "page_num" : i + 1,
            "text" : reader.pages[i].extract_text() or ""
        }
        # Add the text to the list of pages
        pages.append(page)

    return pages

# flatMap : Regroupe tout les pages de chaque pdf dans une seule liste
pages_rdd = rdd.flatMap(extract_pages_from_pdf)

# 2. **Preprocess / clean** noisy PDF text -  cleaned as needed
Common PDF extraction artifacts such as hyphenation across line breaks, spurious line breaks, repeated whitespace, empty pages, headers/footers when they harm retrieval, and duplicated boilerplate. 
All preprocessing and cleaning procedures must be documented in the report.

In [7]:
def clean_text(page_from_pdf: dict) -> dict:
    """Basic cleanup for PDF-extracted text."""
    text = page_from_pdf["text"]
    # 1. Supprimer les césures en fin de ligne
    text = text.replace("-\n", "")
    
    # 2. Remplacer les retours à la ligne restants par des espaces
    text = text.replace("\n", " ")
    
    # 3. Réduire les espaces multiples à un seul espace
    text = re.sub(r"\s+", " ", text)
    
    # 4. Supprimer les espaces en début et fin
    page_from_pdf["text"] = text.strip()
    
    return page_from_pdf

# map : traitement chaque page
clean_rdd = pages_rdd.map(clean_text)

In [8]:
# filter : supprimer les pages vides
#final_rdd = clean_rdd.filter(lambda x: len(x["text"]) > 50)

In [9]:
print(clean_rdd.count(), "pages extraites")
print(clean_rdd.first())

1662 pages extraites
{'pdf_name': '1-intro.pdf', 'page_num': 1, 'text': 'INFO-H515: BIG DATA: DISTRIBUTED MANAGEMENT Lecture 1: Introduction, HDFS, Map/Reduce Dimitris Sacharidis'}


# 3. **Chunk** the text into fixed-size windows with overlap

In [10]:
def word_based_chunk_page(page_from_pdf: dict) -> list:
    """Split one page into overlapping word chunks."""
    
    chunk_size_words = cfg.chunk_size_words 
    overlap_words = cfg.chunk_overlap_words

    words = page_from_pdf["text"].split()
    page_num = page_from_pdf["page_num"]
    
    chunks = []
    step = chunk_size_words - overlap_words
    
    if step <= 0:
        raise ValueError("overlap_words must be smaller than chunk_size_words")
    
    chunk_index = 0
    for start in range(0, len(words), step):
        end = start + chunk_size_words
        chunk_words = words[start:end]
        if not chunk_words:
            break
        chunks.append({
            "chunk_id": f"p{page_num}_c{chunk_index}",
            "pdf_name": page_from_pdf["pdf_name"],
            "page_num":  page_num,
            "text": " ".join(chunk_words)
        })
        chunk_index += 1
        if end >= len(words):
            break
    return chunks
# Build chunks
clean_chunk_rdd = clean_rdd.flatMap(word_based_chunk_page)

In [11]:
print(clean_chunk_rdd.count(), " chunks")
print(clean_chunk_rdd.first())

2982  chunks
{'chunk_id': 'p1_c0', 'pdf_name': '1-intro.pdf', 'page_num': 1, 'text': 'INFO-H515: BIG DATA: DISTRIBUTED MANAGEMENT Lecture 1: Introduction, HDFS, Map/Reduce Dimitris Sacharidis'}


# 4. **Embed**: TF-IDF (baseline) and/or SBERT (dense)

## TF-IDF
TF-IDF calcule les fréquences de termes globales. Il faut collecter tous les chunks avant d'éffectuer se calcule.

In [12]:
def build_tfidf_index(chunks: List[Dict]) -> Tuple[TfidfVectorizer, np.ndarray]:
    """Fit TF-IDF on chunk texts and return (vectorizer, doc_matrix).

    Hints
    -----
    - Collect texts: `texts = [c["text"] for c in chunks]`
    - Use `TfidfVectorizer(...).fit_transform(texts)` to build a sparse matrix `X`
      with shape `[n_chunks, |V|]`.
    - Return `(vectorizer, X)`.
    """
    # YOUR CODE HERE
    texts = [c["text"] for c in chunks]

    vectorizer = TfidfVectorizer()
    X = vectorizer.fit_transform(texts)

    return (vectorizer, X)

# TF-IDF embedding
# Collecter tous les chunks
chunks = clean_chunk_rdd.collect()

# Fit TF-IDF
tf_idf_vectorizer, X_tf_idf = build_tfidf_index(chunks)

print(f"{len(chunks)} chunks")
print(f"Matrice TF-IDF : {X_tf_idf.shape}")  # (n_chunks, vocab_size)

2982 chunks
Matrice TF-IDF : (2982, 18371)


## Word2Vec

In [13]:
def build_word2vec_index(chunks: List[Dict], vector_size: int = 100, 
                          window: int = 5, min_count: int = 2, 
                          epochs: int = 10) -> Tuple[Word2Vec, np.ndarray]:
    """
    Train a Word2Vec model on chunk texts and generate chunk-level embeddings
    by averaging word vectors.
    """
    # Tokeniser chaque chunk en liste de mots
    tokenized = [c["text"].lower().split() for c in chunks]
    
    # Entraîner le modèle Word2Vec
    model = Word2Vec(
        sentences=tokenized,
        vector_size=vector_size,  # dimension des vecteurs
        window=window,            # contexte (mots voisins)
        min_count=min_count,      # ignorer les mots trop rares
        workers=4,                # parallélisme interne gensim
        epochs=epochs
    )
    
    # Générer l'embedding de chaque chunk ( moyenne des vecteurs de ses mots)
    def embed_chunk(words):
        # Garder uniquement les mots connus du modèle
        known = [w for w in words if w in model.wv]
        if not known:
            # vecteur nul pour les chunks vide ou des mots inconnu 
            return np.zeros(vector_size, dtype=np.float32)
        return np.mean([model.wv[w] for w in known], axis=0).astype(np.float32)
    
    embeddings = np.array([embed_chunk(words) for words in tokenized])
    
    return model, embeddings

# Entraîner Word2Vec
w2v_model, w2v_embeddings = build_word2vec_index(chunks)
print(f"Matrice Word2Vec : {w2v_embeddings.shape}")

Matrice Word2Vec : (2982, 100)


## SBERT

De même pour SBERT, on doit l'executer sur l'ensemble des chunks, et non chunk pas chunk.

In [14]:
def build_sbert_index(chunks: List[Dict], model_name: str) -> Tuple[SentenceTransformer, np.ndarray]:
    """Compute dense embeddings for all chunks using a SentenceTransformer.

    What SBERT gives you
    --------------------
    A SentenceTransformer is a Transformer encoder + pooling head that outputs one
    fixed-size embedding per input text. These embeddings are designed so that
    cosine similarity correlates with semantic similarity.

    Hints
    -----
    - `model = SentenceTransformer(model_name)`
    - `emb = model.encode(texts, show_progress_bar=True, normalize_embeddings=True)`
    - Cast to float32: `emb = np.asarray(emb, dtype=np.float32)`
    - Return `(model, emb)` where `emb` has shape `[n_chunks, dim]`.
    """
    # YOUR CODE HERE
    texts = [c["text"] for c in chunks]

    model = SentenceTransformer(model_name)
    emb = model.encode(
        texts,
        show_progress_bar=True,
        normalize_embeddings=True
    )

    emb = np.asarray(emb, dtype=np.float32)

    return model, emb

# SBERT embedding
model_name = "all-MiniLM-L6-v2"  # léger et efficace, 384 dimensions
sbert_model, sbert_emb = build_sbert_index(chunks, model_name)

print(f"Matrice SBERT : {sbert_emb.shape}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/94 [00:00<?, ?it/s]

Matrice SBERT : (2982, 384)


# 5. **Store** a lightweight local index

Pour sauvegarder en paquet, on aura besoin de SparkSession, car c'est lui qui gère les DataFrames / Paquet. 

In [15]:
# Besoin d'une SparkSession pour les DataFrames/Parquet
spark = SparkSession.builder \
    .config(conf=sc.getConf()) \
    .getOrCreate()

def save_to_hdfs_parquet(chunks, embeddings, embedding_strategy, hdfs_output_dir):
    """
    Sauvegarde chunks + embeddings en Parquet sur HDFS au format
    """
    
    schema = StructType([
        StructField("id",                 LongType(),              False),
        StructField("source_pdf",         StringType(),            True),
        StructField("page_num",           IntegerType(),           True),
        StructField("chunk_id",           StringType(),            True),
        StructField("start_word",         StringType(),           True),
        StructField("end_word",           StringType(),           True),
        StructField("chunk_text",         StringType(),            True),
        StructField("embedding",          ArrayType(FloatType()),  True),
        StructField("embedding_strategy", StringType(),            True),
    ])
    
    records = []
    for i, (chunk, emb) in enumerate(zip(chunks, embeddings)):
        words = chunk["text"].split()
        records.append((
            i, # id
            chunk["pdf_name"], # "source_pdf"
            chunk["page_num"], # "page_num"           
            chunk["chunk_id"], # "chunk_id"
            words[0], #"start_word"
            words[-1], #"end_word"
            chunk["text"], #"chunk_text"
            [float(v) for v in emb], #"embedding"
            embedding_strategy, #"embedding_strategy"
        ))

    # Redistribuer dans Spark et sauvegarder en Parquet sur HDFS
    df = spark.createDataFrame(records, schema=schema)
    df.write.mode("overwrite").parquet(f"hdfs://namenode:8020/{hdfs_output_dir}")
    print(f"Sauvegardé : hdfs://namenode:8020/{hdfs_output_dir}")
    print(f"Schema :") 
    df.printSchema()

In [16]:
# Sauvegarde tfidf prend un peu trop de temps
#save_to_hdfs_parquet(chunks, X_tf_idf.toarray(), "tfidf",  "project/embeddings/tfidf") 

In [17]:
# Sauvegarde word2vec
#save_to_hdfs_parquet(chunks, w2v_embeddings,   "word2vec","project/embeddings/word2vec")

In [18]:
# Sauvegarde sbert
save_to_hdfs_parquet(chunks, sbert_emb, "sbert",  "project/embeddings/sbert")

Sauvegardé : hdfs://namenode:8020/project/embeddings/sbert
Schema :
root
 |-- id: long (nullable = false)
 |-- source_pdf: string (nullable = true)
 |-- page_num: integer (nullable = true)
 |-- chunk_id: string (nullable = true)
 |-- start_word: string (nullable = true)
 |-- end_word: string (nullable = true)
 |-- chunk_text: string (nullable = true)
 |-- embedding: array (nullable = true)
 |    |-- element: float (containsNull = true)
 |-- embedding_strategy: string (nullable = true)



6. **Retrieve** Top-*k* chunks for a query using similarity search

In [22]:
def cosine_similarity(query_vec: np.ndarray, doc_matrix: np.ndarray) -> np.ndarray:
    """Compute cosine similarity between query_vec and each row of doc_matrix."""
    # YOUR CODE HERE
    # Produit scalaire entre la requête et chaque document
    dot_products = doc_matrix @ query_vec

    # Normes
    query_norm = np.linalg.norm(query_vec)
    doc_norms = np.linalg.norm(doc_matrix, axis=1)

    # Pour éviter une division par zéro
    eps = 1e-10
    similarities = dot_products / (doc_norms * query_norm + eps)

    return similarities


def euclidean_distance(query_vec: np.ndarray, doc_matrix: np.ndarray) -> np.ndarray:
    """Compute Euclidean distance between query_vec and each row of doc_matrix."""
    # YOUR CODE HERE
    # S'assurer que query_vec est 1D
    query_vec = query_vec.reshape(1, -1)

    # Différence vectorisée : (n_docs, dim)
    diff = doc_matrix - query_vec

    # Distance euclidienne par document
    distances = np.sqrt(np.sum(diff ** 2, axis=1))

    return distances


def retrieve_top_k(
    query: str,
    chunks: List[Dict],
    doc_matrix,
    k: int,
    embedding_strategy: str,
    similarity_metric: str,
    tfidf_vectorizer: Optional[TfidfVectorizer] = None,
    sbert_model: Optional[SentenceTransformer] = None,
    w2v_model: Optional[SentenceTransformer] = None,
) -> List[Dict]:
    """Return Top-k retrieved chunks for the query."""
    # -----------------------------
    # 1. Embed the query
    # -----------------------------
    if embedding_strategy == "tfidf":
        if tf_idf_vectorizer is None:
            raise ValueError("tfidf_vectorizer must be provided for TF-IDF retrieval")
        query_vec = tfidf_vectorizer.transform([query]).toarray()[0]

    elif embedding_strategy == "sbert":
        if sbert_model is None:
            raise ValueError("sbert_model must be provided for SBERT retrieval")
        query_vec = sbert_model.encode(
            query,
            normalize_embeddings=True
        )
        query_vec = np.asarray(query_vec, dtype=np.float32)
    
    elif embedding_strategy == "word2vec":
        if w2v_model is None:
            raise ValueError("w2v_model must be provided for Word2Vec retrieval")
        words = query.lower().split()
        known = [w for w in words if w in w2v_model.wv]
        if not known:
            raise ValueError("No known words in query")
        query_vec = np.mean(
            [w2v_model.wv[w] for w in known], axis=0
        ).astype(np.float32)
    
    else:
        raise ValueError(f"Unknown embedding_strategy: {embedding_strategy}")

    # -----------------------------
    # 2. Compute similarity / distance
    # -----------------------------
    if similarity_metric == "cosine":
        scores = cosine_similarity(query_vec, doc_matrix)
        # higher is better
        top_indices = np.argsort(scores)[::-1][:k]

    elif similarity_metric == "euclidean":
        scores = euclidean_distance(query_vec, doc_matrix)
        # lower is better
        top_indices = np.argsort(scores)[:k]

    else:
        raise ValueError(f"Unknown similarity_metric: {similarity_metric}")

    # -----------------------------
    # 3. Package results
    # -----------------------------
    results = []
    for idx in top_indices:
        results.append({
            "chunk": chunks[idx],
            "score": float(scores[idx])
        })

    return results

In [23]:
query = "What is Big Data ?"

# Avec SBERT + cosine
results = retrieve_top_k(
    query=query,
    chunks=chunks,
    doc_matrix=sbert_emb,
    k=5,
    embedding_strategy="sbert",
    similarity_metric="cosine",
    sbert_model=sbert_model
)

for r in results:
    print(f"Score: {r['score']:.3f} | {r['chunk']['pdf_name']} p.{r['chunk']['page_num']}")
    print(f"  {r['chunk']['text'][:150]}...")
    print()

Score: 1.000 | 1-intro.pdf p.3
  WHAT IS BIG DATA?...

Score: 0.795 | 1-intro.pdf p.15
  BIG DATA: SOME DEFINITIONS (1/4) Big Data is a term encompassing the use of techniques to capture, process, analyse and visualize potentially large da...

Score: 0.782 | 1-intro.pdf p.17
  BIG DATA: SOME DEFINITIONS (3/4) Big data are data sets that are so voluminous and complex that traditional data processing application software are i...

Score: 0.778 | 1-intro.pdf p.18
  BIG DATA: SOME DEFINITIONS (4/4) Big data refers to extremely large datasets that cannot be easily managed, processed, or analyzed using traditional d...

Score: 0.737 | a_TowardScalableSystemsForBigDataAnalytics.pdf p.3
  metric. However, it incorporates an evolutionary aspect in the deﬁnition (over time or across sectors) of what a dataset must be to be considered as b...



7. **Construct prompts** that ground the model and request citations

8. **Generate** questions with a free local model

9. **Evaluate** retrieval and generation with a tiny gold set + manual inspection